## 01 - Pitt - Preprocessing

We have new Pitt dataset from DementiaBank. We will extract the transcribed text and export to a csv file

In [1]:
# import required packages
import pandas as pd
import os
import re

In [2]:
# labels
# AD = Alzheimer's Disease (positive class)
# CN = Cognitively Normal (negative class)

In [3]:
# please point to correct downloaded transcribed folders of the original dataset
data_path = "../../../data/" 
path_ad = data_path + "TalkBank downloads/Pitt/Dementia/Cookie/"
path_cn = data_path + "TalkBank downloads/Pitt/Control/Cookie/"

In [4]:
data = {"ad":[], 
        "cn":[]
       }

file_names = {"ad":[],
              "cn":[]
             }

In [5]:
def line_clean(line):
    """
        Clean line from extra information like "*PAR: ", "\x1530_5640\x15", etc
    """
    result = line
    if result.startswith("*PAR"):
        result = result.replace("*PAR:\t", "")
        result = re.sub(".*", "", result)
        result = re.sub("\[.*\]", "", result)
        result = result.strip("&-+<").strip()
    else:
        result = ""
    return result

s = '*PAR:\tthe scene is <in the> [/] in the kitchen . \x1530_5640\x15\n'
assert line_clean(s) == "the scene is <in the>  in the kitchen ." 

In [6]:
def read_data(path):
    """
        get data from "txt" files from a specified folder (ad/cn)
        returns list of text (strings)
    """
    result = []
    filenames = []

    # get only txt files
    files = [item
             for item in os.listdir(path)
             if item.endswith("cha")
            ]
    
    for fname in files:
        print("Processing files =", fname) #, end="\r")
        with open(path + fname, encoding="utf-8") as f:
            lines = f.readlines()
            print("Line counts =", len(lines)) #, end="\r")
            # apply filter to clean text
            lines = [line_clean(line)
                     for line in lines
                     if line.startswith("*PAR")
                    ]
            print("After cleaning, Line counts =", len(lines)) #, end="\r")

            text = "\n".join(lines)
            result.append(text)
            filenames.append(fname.replace(".cha", "")) # remove file extensions

    return (result, filenames)



In [7]:
data["ad"], file_names["ad"] = read_data(path_ad)
data["cn"], file_names["cn"] = read_data(path_cn)

Processing files = 001-0.cha
Line counts = 85
After cleaning, Line counts = 10
Processing files = 001-2.cha
Line counts = 125
After cleaning, Line counts = 13
Processing files = 003-0.cha
Line counts = 172
After cleaning, Line counts = 23
Processing files = 005-0.cha
Line counts = 73
After cleaning, Line counts = 9
Processing files = 005-2.cha
Line counts = 92
After cleaning, Line counts = 7
Processing files = 007-1.cha
Line counts = 105
After cleaning, Line counts = 13
Processing files = 007-3.cha
Line counts = 85
After cleaning, Line counts = 9
Processing files = 010-0.cha
Line counts = 186
After cleaning, Line counts = 22
Processing files = 010-1.cha
Line counts = 201
After cleaning, Line counts = 27
Processing files = 010-2.cha
Line counts = 208
After cleaning, Line counts = 32
Processing files = 010-3.cha
Line counts = 128
After cleaning, Line counts = 17
Processing files = 010-4.cha
Line counts = 92
After cleaning, Line counts = 9
Processing files = 014-2.cha
Line counts = 161
Af

In [8]:
# double check data
assert len(data["ad"]) == 306
assert len(data["cn"]) == 243
assert data["ad"][0].startswith("mhm")
assert data["ad"][1].startswith("mhm")
assert data["cn"][0].startswith("the scene is <in the>")
assert data["cn"][1].startswith("oh I see the sink is running over")
print("Passed")

Passed


In [9]:
# assert filenames
assert file_names["ad"][0].startswith("001-0")
assert file_names["ad"][1].startswith("001-2")
assert file_names["ad"][-1].startswith("714-0")

assert file_names["cn"][0].startswith("002-0")
assert file_names["cn"][1].startswith("002-1")
assert file_names["cn"][-1].startswith("709-2")

print("Passed")

Passed


In [10]:
# take a look at the data
print(data["ad"][0][:100])

mhm .
alright .
there's &-um a young boy that's getting a cookie jar .
and it  he's &-uh in bad shap


In [11]:
# take a look at the data
print(data["cn"][0][:100])

the scene is <in the>  in the kitchen .
the mother is wiping dishes and the water is running on the 


In [12]:
print(file_names["ad"][0:5])

['001-0', '001-2', '003-0', '005-0', '005-2']


In [13]:
print(file_names["cn"][0:5])

['002-0', '002-1', '002-2', '002-3', '006-2']


In [14]:
# combine data into a dataframe, for export to a csv for further processing
data_dict = {
    "file_id": [],  #  file name
    "text": [],    # the transcript
    "target": []   # the target: positive (AD), negative (CN)
}
for item, fname in zip(data["ad"], file_names["ad"]):
    data_dict["file_id"].append(fname)
    data_dict["text"].append(item)
    data_dict["target"].append("AD")

for item, fname in zip(data["cn"], file_names["cn"]):
    data_dict["file_id"].append(fname)
    data_dict["text"].append(item)
    data_dict["target"].append("CN")

print("Text count =", len(data_dict["text"]))
print("Target count =", len(data_dict["target"]))
print("File_id count =", len(data_dict["target"]))

Text count = 549
Target count = 549
File_id count = 549


In [15]:
# create a dataframe to export to csv
data_df = pd.DataFrame(data_dict)
print(len(data_df))
data_df.head()

549


,file_id,text,target
0,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD
1,001-2,mhm .\nthere's a young boy &-uh going in a coo...,AD
2,003-0,here's a cookie jar .\nand the lid is off the ...,AD
3,005-0,the boy is slipping off the stool .\nhe's tryi...,AD
4,005-2,okay ‡ he's fallin(g) off a chair .\nshe's &-...,AD


In [16]:
data_df.describe()

,file_id,text,target
count,549,549,549
unique,549,549,2
top,001-0,mhm .\nalright .\nthere's &-um a young boy tha...,AD
freq,1,1,306


We can see we have total of 549 examples, with 306 (about 55%) positive (AD) and 243 (about 45%) negative (CN). The dataset is relatively balanced, although there are slightly more examples with positive labels. 

### Create CSV output 

In [17]:
# write to csv
print("Writing to csv files...")
path = data_path + "pitt/"
data_df.to_csv(path + "text_pitt.csv", index=False)
print("Completed")

Writing to csv files...
Completed
